# Phase 1: Data Acquisition & Inspection

**Project:** Bureau of Tax and Economic Analysis — Quantitative Research

**Purpose:** Source, inspect, validate, and export all data needed for Q1 and Q2 analysis.

**Data Sources:**
1. NYS DOL CES data (`ces.csv` from https://dol.ny.gov/statistics-ceszip)
2. NYC OpenData 311 Service Requests (API at https://data.cityofnewyork.us/resource/erm2-nwe9.json)

In [1]:
import pandas as pd
import requests
import json
import os
from datetime import datetime

DATA_DIR = '../data'
print(f'Data directory: {DATA_DIR}')
print(f'Files in data directory: {os.listdir(DATA_DIR)}')
print(f'Notebook run time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

Data directory: ../data
Files in data directory: ['ces_minor.csv', 'ces_hours.csv', 'CES_Readme.txt', 'ces.csv', 'ces_earnings.csv', '311_complaints.csv', 'ces.zip', 'ces_hourearn.csv']
Notebook run time: 2026-04-20 16:11:53


---
## Step 1.1–1.2: Load and Inspect CES Data

In [2]:
ces = pd.read_csv(f'{DATA_DIR}/ces.csv')
print(f'ces.csv shape: {ces.shape}')
print(f'\nColumns: {list(ces.columns)}')
print(f'\nData types:')
print(ces.dtypes)
print(f'\nFirst 3 rows:')
ces.head(3)

ces.csv shape: (23810, 19)

Columns: ['SERIESCODE', 'INDUSTRY_TITLE', 'AREATYPE', 'AREA', 'AREANAME', 'YEAR', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'ANNUAL  ']

Data types:
SERIESCODE          int64
INDUSTRY_TITLE        str
AREATYPE            int64
AREA                int64
AREANAME              str
YEAR                int64
JAN                 int64
FEB                 int64
MAR               float64
APR               float64
MAY               float64
JUN               float64
JUL               float64
AUG               float64
SEP               float64
OCT               float64
NOV               float64
DEC               float64
ANNUAL                str
dtype: object

First 3 rows:


,SERIESCODE,INDUSTRY_TITLE,AREATYPE,AREA,AREANAME,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL
0,0,Total Nonfarm,1,36,New York State,1990,8093100,8121400,8180500.0,8187500.0,8283200.0,8333300.0,8202400.0,8206000.0,8213500.0,8211100.0,8211200.0,8201800.0,8203800 ...
1,0,Total Nonfarm,1,36,New York State,1991,7841400,7839000,7870500.0,7878900.0,7934800.0,7986600.0,7833400.0,7834400.0,7844300.0,7886700.0,7904100.0,7888700.0,7878600 ...
2,0,Total Nonfarm,1,36,New York State,1992,7597100,7611300,7648100.0,7701800.0,7762200.0,7803900.0,7724000.0,7713500.0,7726100.0,7781200.0,7790500.0,7807400.0,7722300 ...


### README Reference

Per `CES_Readme.txt`:
- `SERIESCODE` = CES published line code
- `INDUSTRY_TITLE` = CES published line name
- `AREATYPE`: 01=state, 21=MSA, 23=MD (Metro Division)
- `AREA` = Numeric metro area FIPS code
- `AREANAME` = Name of area
- `YEAR` = Year
- `JAN`–`DEC` = Monthly employment
- `ANNUAL` = Summed monthly employment / 12
- **Data is NOT seasonally adjusted** — compare same month across years only
- **Values appear to be in actual units** (not thousands) based on NYS total ~8M

---
## Step 1.3: Verify Geography — How is NYC Represented?

In [3]:
print('=== ALL UNIQUE AREANAMES ===')
for name in sorted(ces['AREANAME'].unique()):
    print(f'  {name}')

print(f'\n=== ANSWER: "New York City" is a direct AREANAME entry ===')
print('This is NOT the broader MSA (which would include NJ/PA suburbs).')
print('This resolves Gotcha G1: we can filter AREANAME == "New York City" directly.')

=== ALL UNIQUE AREANAMES ===
  Albany-Schenectady-Troy Metro Area
  Binghamton Metro Area
  Buffalo-Cheektowaga Metro Area
  Elmira Metro Area
  Glens Falls Metro Area
  Ithaca Metro Area
  Kingston Metro Area
  Kiryas Joel-Poughkeepsie-Newburgh NY
  Nassau-Suffolk Metropolitan Division
  New York City
  New York State
  Rochester Metro Area
  Syracuse Metro Area
  Utica-Rome Metro Area
  Watertown-Fort Drum Metro Area

=== ANSWER: "New York City" is a direct AREANAME entry ===
This is NOT the broader MSA (which would include NJ/PA suburbs).
This resolves Gotcha G1: we can filter AREANAME == "New York City" directly.


In [4]:
# Sanity check: NYC total nonfarm employment
nyc_latest = ces[(ces['AREANAME'] == 'New York City') & (ces['YEAR'] == ces['YEAR'].max())]
total_nf = nyc_latest[nyc_latest['INDUSTRY_TITLE'] == 'Total Nonfarm']
print('=== NYC Total Nonfarm Employment (latest year) ===')
print(total_nf[['YEAR', 'JAN', 'FEB']].to_string(index=False))
print(f'\nJan 2026: {total_nf["JAN"].values[0]:,.0f}')
print(f'Feb 2026: {total_nf["FEB"].values[0]:,.0f}')
print('\nSanity check: NYC total nonfarm ~4.5-4.8M is plausible for the 5 boroughs.')
print('The broader MSA would be ~10M. Our figure is clearly NYC proper.')

=== NYC Total Nonfarm Employment (latest year) ===
 YEAR     JAN     FEB
 2026 4775100 4791000

Jan 2026: 4,775,100
Feb 2026: 4,791,000

Sanity check: NYC total nonfarm ~4.5-4.8M is plausible for the 5 boroughs.
The broader MSA would be ~10M. Our figure is clearly NYC proper.


---
## Step 1.4: Verify Latest Available Month

In [5]:
print(f'Max year in data: {ces["YEAR"].max()}')
print(f'Download date: 2026-04-20')
print()

latest_year = ces[ces['YEAR'] == ces['YEAR'].max()]
print(f'Months with non-null data in {ces["YEAR"].max()}:')
for m in ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']:
    non_null = latest_year[m].notna().sum()
    status = 'HAS DATA' if non_null > 0 else 'NO DATA'
    print(f'  {m}: {non_null:>5} rows [{status}]')

print(f'\n=== ANSWER: Latest available month is FEBRUARY 2026 ===')
print('This means:')
print('  - YoY comparison: Feb 2026 vs Feb 2025')
print('  - 5-year comparison: Feb 2026 vs Feb 2021')
print('  - CRITICAL: Feb 2021 was still in COVID disruption')

Max year in data: 2026
Download date: 2026-04-20

Months with non-null data in 2026:
  JAN:   650 rows [HAS DATA]
  FEB:   650 rows [HAS DATA]
  MAR:     0 rows [NO DATA]
  APR:     0 rows [NO DATA]
  MAY:     0 rows [NO DATA]
  JUN:     0 rows [NO DATA]
  JUL:     0 rows [NO DATA]
  AUG:     0 rows [NO DATA]
  SEP:     0 rows [NO DATA]
  OCT:     0 rows [NO DATA]
  NOV:     0 rows [NO DATA]
  DEC:     0 rows [NO DATA]

=== ANSWER: Latest available month is FEBRUARY 2026 ===
This means:
  - YoY comparison: Feb 2026 vs Feb 2025
  - 5-year comparison: Feb 2026 vs Feb 2021
  - CRITICAL: Feb 2021 was still in COVID disruption


---
## Step 1.5: Identify 2-Digit NAICS Industry Mapping

The CES data uses BLS industry codes, not NAICS codes directly. However, the industry titles map to 2-digit NAICS sectors.

**Note:** Some 2-digit NAICS sectors are combined in the CES data for NYC:
- Mining (NAICS 21) + Construction (NAICS 23) = "Mining, Logging and Construction" (combined)
- Education (NAICS 61) is "Private Educational Services" only (excludes public education)
- Health Care (NAICS 62) is private sector only
- Government (NAICS 92) is shown separately

In [6]:
# Define the 2-digit NAICS to BLS CES industry mapping for NYC
# Using the most granular level available that maps to individual 2-digit NAICS sectors

naics_2digit_mapping = {
    '21+23': 'Mining, Logging and Construction',
    '31-33': 'Manufacturing',
    '42': 'Wholesale Trade',
    '44-45': 'Retail Trade',
    '48-49': 'Transportation and Warehousing',
    '22': 'Utilities',
    '51': 'Information',
    '52': 'Finance and Insurance',
    '53': 'Real Estate and Rental and Leasing',
    '54': 'Professional, Scientific, and Technical Services',
    '55': 'Management of Companies and Enterprises',
    '56': 'Administrative and Support and Waste Management and Remediation Services',
    '61': 'Private Educational Services',
    '62': 'Health Care and Social Assistance',
    '71': 'Arts, Entertainment, and Recreation',
    '72': 'Accommodation and Food Services',
    '81': 'Other Services',
    '92': 'Government',
}

print('=== 2-DIGIT NAICS → CES INDUSTRY TITLE MAPPING ===')
print(f'{"NAICS":<8} {"CES Industry Title":<65} {"In Data?":<10}')
print('-' * 83)

nyc_industries = set(ces[ces['AREANAME'] == 'New York City']['INDUSTRY_TITLE'].unique())

for naics, title in naics_2digit_mapping.items():
    found = title in nyc_industries
    print(f'{naics:<8} {title:<65} {"YES" if found else "NO":<10}')

print(f'\nNote: NAICS 21+23 (Mining + Construction) are COMBINED in CES data for NYC.')
print('Note: NAICS 11 (Agriculture) is excluded from CES data entirely.')
print('Note: Education (61) and Health Care (62) are PRIVATE SECTOR only.')

=== 2-DIGIT NAICS → CES INDUSTRY TITLE MAPPING ===
NAICS    CES Industry Title                                                In Data?  
-----------------------------------------------------------------------------------
21+23    Mining, Logging and Construction                                  YES       
31-33    Manufacturing                                                     YES       
42       Wholesale Trade                                                   YES       
44-45    Retail Trade                                                      YES       
48-49    Transportation and Warehousing                                    YES       
22       Utilities                                                         YES       
51       Information                                                       YES       
52       Finance and Insurance                                             YES       
53       Real Estate and Rental and Leasing                                YES       
54   

In [7]:
# Verify: extract Feb 2026 and Feb 2025 employment for each mapped industry
nyc = ces[ces['AREANAME'] == 'New York City']

latest = nyc[nyc['YEAR'] == 2026]
prior = nyc[nyc['YEAR'] == 2025]

print('=== SPOT CHECK: Feb 2026 vs Feb 2025 for each 2-digit NAICS sector ===')
print(f'{"NAICS":<8} {"Industry":<65} {"Feb 2026":>10} {"Feb 2025":>10}')
print('-' * 95)

for naics, title in naics_2digit_mapping.items():
    feb26 = latest[latest['INDUSTRY_TITLE'] == title]['FEB'].values
    feb25 = prior[prior['INDUSTRY_TITLE'] == title]['FEB'].values
    v26 = f'{feb26[0]:,.0f}' if len(feb26) > 0 and pd.notna(feb26[0]) else 'N/A'
    v25 = f'{feb25[0]:,.0f}' if len(feb25) > 0 and pd.notna(feb25[0]) else 'N/A'
    print(f'{naics:<8} {title:<65} {v26:>10} {v25:>10}')

=== SPOT CHECK: Feb 2026 vs Feb 2025 for each 2-digit NAICS sector ===
NAICS    Industry                                                            Feb 2026   Feb 2025
-----------------------------------------------------------------------------------------------
21+23    Mining, Logging and Construction                                     130,400    137,000
31-33    Manufacturing                                                         50,200     53,300
42       Wholesale Trade                                                      129,700    132,700
44-45    Retail Trade                                                         294,100    296,800
48-49    Transportation and Warehousing                                       132,500    134,100
22       Utilities                                                             17,000     16,800
51       Information                                                          221,300    217,100
52       Finance and Insurance                           

In [8]:
# Also check 5-year lookback: Feb 2021
five_yr = nyc[nyc['YEAR'] == 2021]

print('=== SPOT CHECK: Feb 2021 (5-year lookback baseline) ===')
print(f'{"NAICS":<8} {"Industry":<65} {"Feb 2026":>10} {"Feb 2021":>10}')
print('-' * 95)

for naics, title in naics_2digit_mapping.items():
    feb26 = latest[latest['INDUSTRY_TITLE'] == title]['FEB'].values
    feb21 = five_yr[five_yr['INDUSTRY_TITLE'] == title]['FEB'].values
    v26 = f'{feb26[0]:,.0f}' if len(feb26) > 0 and pd.notna(feb26[0]) else 'N/A'
    v21 = f'{feb21[0]:,.0f}' if len(feb21) > 0 and pd.notna(feb21[0]) else 'N/A'
    print(f'{naics:<8} {title:<65} {v26:>10} {v21:>10}')

print(f'\nCRITICAL CONTEXT: Feb 2021 was still in COVID disruption.')
print('This means the 5-year comparison captures the FULL post-COVID recovery arc.')

=== SPOT CHECK: Feb 2021 (5-year lookback baseline) ===
NAICS    Industry                                                            Feb 2026   Feb 2021
-----------------------------------------------------------------------------------------------
21+23    Mining, Logging and Construction                                     130,400    135,700
31-33    Manufacturing                                                         50,200     52,100
42       Wholesale Trade                                                      129,700    119,600
44-45    Retail Trade                                                         294,100    283,100
48-49    Transportation and Warehousing                                       132,500    114,500
22       Utilities                                                             17,000     14,300
51       Information                                                          221,300    210,300
52       Finance and Insurance                                          

---
## Step 1.6–1.7: Explore 311 API Schema

Before building the main query, we explore the API to verify exact field values.

In [9]:
API_BASE = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'

# Step 1.6: Verify exact agency_name values
params = {
    '$select': 'agency_name',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59'",
    '$group': 'agency_name',
    '$limit': 100
}
r = requests.get(API_BASE, params=params)
agencies = sorted([row['agency_name'] for row in r.json()])

print('=== ALL agency_name values in date range ===')
for a in agencies:
    print(f'  {a}')

print(f'\n=== VERIFICATION ===')
target_agencies = [
    'New York City Police Department',
    'Department of Housing Preservation and Development'
]
for t in target_agencies:
    match = t in agencies
    print(f'  "{t}" found in API: {match}')

=== ALL agency_name values in date range ===
  Department for the Aging
  Department of Buildings
  Department of Consumer and Worker Protection
  Department of Education
  Department of Environmental Protection
  Department of Health and Mental Hygiene
  Department of Homeless Services
  Department of Housing Preservation and Development
  Department of Information Technology and Telecommunications
  Department of Parks and Recreation
  Department of Sanitation
  Department of Transportation
  Economic Development Corporation
  Mayor's Office of Special Enforcement
  New York City Police Department
  Office of Technology and Innovation
  Operations Unit - Department of Homeless Services
  Taxi and Limousine Commission

=== VERIFICATION ===
  "New York City Police Department" found in API: True
  "Department of Housing Preservation and Development" found in API: True


In [10]:
# Step 1.7: Explore complaint_type values for noise and illegal parking
params = {
    '$select': 'complaint_type, count(*) as cnt',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' AND (complaint_type like '%Noise%' OR complaint_type like '%Illegal Parking%')",
    '$group': 'complaint_type',
    '$order': 'cnt DESC',
    '$limit': 100
}
r = requests.get(API_BASE, params=params)
complaints = r.json()

print('=== NOISE & PARKING complaint_type values in date range ===')
total_noise = 0
exact_noise = 0
parking = 0
print(f'{"complaint_type":<55} {"count":>10}')
print('-' * 67)
for row in complaints:
    ct = row['complaint_type']
    cnt = int(row['cnt'])
    print(f'{ct:<55} {cnt:>10,}')
    if 'Noise' in ct:
        total_noise += cnt
        if ct == 'Noise':
            exact_noise = cnt
    if ct == 'Illegal Parking':
        parking = cnt

print(f'\n{"TOTAL (all Noise subcategories)":<55} {total_noise:>10,}')
print(f'\n=== CRITICAL FINDING ===')
print(f'Exact match "Noise":           {exact_noise:>10,}')
print(f'All noise subcategories:        {total_noise:>10,}')
print(f'Difference:                     {total_noise - exact_noise:>10,}')
print(f'\nAn exact match for "Noise" captures only {exact_noise/total_noise*100:.1f}% of noise complaints.')
print(f'This is the trap: most noise complaints are in subcategories like "Noise - Residential".')

=== NOISE & PARKING complaint_type values in date range ===
complaint_type                                               count
-------------------------------------------------------------------
Illegal Parking                                             86,493
Noise - Residential                                         75,543
Noise - Street/Sidewalk                                     12,743
Noise - Vehicle                                             11,372
Noise - Commercial                                          11,114
Noise                                                       10,789
Noise - Helicopter                                           5,290
Noise - Park                                                   498
Noise - House of Worship                                       395

TOTAL (all Noise subcategories)                            127,744

=== CRITICAL FINDING ===
Exact match "Noise":               10,789
All noise subcategories:           127,744
Difference:            

### Noise Matching Decision

**The project says: `complaint_type = noise or illegal parking`**

**Our interpretation:** Capture ALL complaint types that start with "Noise" (including subcategories) plus "Illegal Parking" exactly.

**Justification:**
1. The 311 system categorizes noise complaints into subcategories (Residential, Commercial, Vehicle, etc.)
2. An exact match for "Noise" alone captures only a fraction of noise complaints
3. The project's intent is clearly to analyze noise complaints broadly — excluding subcategories would be misleading
4. We will present results both ways (all subcategories vs. exact match) for transparency

### Critical Discovery: Agency Ownership of Noise Complaints

Before building the main query, we must understand WHICH agencies handle which noise types.

In [11]:
# Which agencies handle noise complaints?
params = {
    '$select': 'agency_name, complaint_type, count(*) as cnt',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' AND starts_with(complaint_type, 'Noise')",
    '$group': 'agency_name, complaint_type',
    '$order': 'cnt DESC',
    '$limit': 100
}
r = requests.get(API_BASE, params=params)
agency_noise = r.json()

print('=== AGENCY × NOISE COMPLAINT TYPE MATRIX ===')
print(f'{"agency_name":<60} {"complaint_type":<30} {"count":>10}')
print('-' * 102)
for row in sorted(agency_noise, key=lambda x: int(x['cnt']), reverse=True):
    print(f'{row["agency_name"]:<60} {row["complaint_type"]:<30} {int(row["cnt"]):>10,}')

print('\n=== KEY FINDINGS ===')
print('1. NYPD handles: Noise - Residential, Noise - Street/Sidewalk, Noise - Vehicle,')
print('   Noise - Commercial, Noise - Park, Noise - House of Worship')
print('2. Department of Environmental Protection (DEP) handles: exact "Noise" type (10,789)')
print('3. Economic Development Corporation handles: Noise - Helicopter (5,290)')
print('4. HPD has ZERO noise complaints — HPD handles housing quality (heat/hot water, plumbing, etc.)')
print('\nIMPLICATION: When we filter for (NYPD OR HPD) AND (noise OR illegal parking),')
print('HPD will return ZERO rows because HPD does not handle noise or parking complaints.')

=== AGENCY × NOISE COMPLAINT TYPE MATRIX ===
agency_name                                                  complaint_type                      count
------------------------------------------------------------------------------------------------------
New York City Police Department                              Noise - Residential                75,543
New York City Police Department                              Noise - Street/Sidewalk            12,743
New York City Police Department                              Noise - Vehicle                    11,372
New York City Police Department                              Noise - Commercial                 11,114
Department of Environmental Protection                       Noise                              10,789
Economic Development Corporation                             Noise - Helicopter                  5,290
New York City Police Department                              Noise - Park                          498
New York City Police Departm

In [12]:
# Verify: what complaint types DOES HPD handle in this period?
params = {
    '$select': 'complaint_type, count(*) as cnt',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' AND agency_name='Department of Housing Preservation and Development'",
    '$group': 'complaint_type',
    '$order': 'cnt DESC',
    '$limit': 20
}
r = requests.get(API_BASE, params=params)
hpd_types = r.json()

print('=== HPD COMPLAINT TYPES (top 20, date range Dec 2021 - Mar 2022) ===')
for row in hpd_types:
    print(f'  {row["complaint_type"]:45s} {int(row["cnt"]):>10,}')

print('\nHPD handles housing quality issues: heat/hot water, plumbing, paint, etc.')
print('HPD has NO noise or illegal parking complaints in this period.')
print('This is NOT a data error — it reflects the actual division of responsibilities between NYC agencies.')

=== HPD COMPLAINT TYPES (top 20, date range Dec 2021 - Mar 2022) ===
  HEAT/HOT WATER                                   121,072
  UNSANITARY CONDITION                              22,267
  PLUMBING                                          15,738
  PAINT/PLASTER                                     13,186
  DOOR/WINDOW                                       11,402
  WATER LEAK                                         9,568
  GENERAL                                            7,723
  ELECTRIC                                           6,869
  FLOORING/STAIRS                                    6,150
  APPLIANCE                                          4,876
  SAFETY                                             3,233
  ELEVATOR                                             516
  OUTSIDE BUILDING                                     201

HPD handles housing quality issues: heat/hot water, plumbing, paint, etc.
HPD has NO noise or illegal parking complaints in this period.
This is NOT a data error —

---
## Step 1.8: Build Full 311 Query and Pull Data

In [13]:
# Build the exact SoQL query URL
# Requirements from Q2a:
#   1. created_date between 12/15/2021 and 3/15/2022 (inclusive)
#   2. agency_name = NYPD or HPD
#   3. complaint_type starts with 'Noise' OR = 'Illegal Parking'

where_clause = (
    "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' "
    "AND (agency_name='New York City Police Department' "
    "OR agency_name='Department of Housing Preservation and Development') "
    "AND (starts_with(complaint_type, 'Noise') "
    "OR complaint_type='Illegal Parking')"
)

select_fields = 'unique_key,created_date,agency_name,complaint_type,descriptor,borough,incident_zip,latitude,longitude'

# The exact query URL
QUERY_URL = (
    f'https://data.cityofnewyork.us/resource/erm2-nwe9.json'
    f'?$select={select_fields}'
    f'&$where={where_clause}'
    f'&$limit=500000'
)

print('=== EXACT QUERY URL ===')
print(QUERY_URL)
print(f'\nURL length: {len(QUERY_URL)} characters')

=== EXACT QUERY URL ===
https://data.cityofnewyork.us/resource/erm2-nwe9.json?$select=unique_key,created_date,agency_name,complaint_type,descriptor,borough,incident_zip,latitude,longitude&$where=created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' AND (agency_name='New York City Police Department' OR agency_name='Department of Housing Preservation and Development') AND (starts_with(complaint_type, 'Noise') OR complaint_type='Illegal Parking')&$limit=500000

URL length: 452 characters


In [14]:
# Execute the query
params = {
    '$select': select_fields,
    '$where': where_clause,
    '$limit': 500000
}

print('Pulling data from NYC OpenData API...')
r = requests.get(API_BASE, params=params)
print(f'Status code: {r.status_code}')

data = r.json()
df_311 = pd.DataFrame(data)
print(f'Rows pulled: {len(df_311):,}')
print(f'Limit was: {500000:,}')
print(f'Truncated: {"YES - WARNING!" if len(df_311) >= 500000 else "No"}')
print(f'\nColumns: {list(df_311.columns)}')

Pulling data from NYC OpenData API...


Status code: 200


Rows pulled: 198,158
Limit was: 500,000
Truncated: No

Columns: ['unique_key', 'created_date', 'agency_name', 'complaint_type', 'descriptor', 'borough', 'incident_zip', 'latitude', 'longitude']


In [15]:
# Step 1.9: Validate the pulled data

print('=== DATA VALIDATION ===')
print(f'Total rows: {len(df_311):,}')
print()

print('Date range:')
print(f'  Min created_date: {df_311["created_date"].min()}')
print(f'  Max created_date: {df_311["created_date"].max()}')
print()

print('Unique agency_name values:')
for a in sorted(df_311['agency_name'].unique()):
    count = len(df_311[df_311['agency_name'] == a])
    print(f'  {a}: {count:,}')
print()

print('Unique complaint_type values:')
for ct in sorted(df_311['complaint_type'].unique()):
    count = len(df_311[df_311['complaint_type'] == ct])
    print(f'  {ct}: {count:,}')
print()

# Duplicate check
dupes = df_311['unique_key'].duplicated().sum()
print(f'Duplicate unique_key values: {dupes}')

# Null checks
print(f'\nNull values per column:')
for col in df_311.columns:
    nulls = df_311[col].isna().sum()
    print(f'  {col}: {nulls:,}')

=== DATA VALIDATION ===
Total rows: 198,158

Date range:
  Min created_date: 2021-12-15T00:02:21.000
  Max created_date: 2022-03-15T23:59:38.000

Unique agency_name values:
  New York City Police Department: 198,158

Unique complaint_type values:
  Illegal Parking: 86,493
  Noise - Commercial: 11,114
  Noise - House of Worship: 395
  Noise - Park: 498
  Noise - Residential: 75,543


  Noise - Street/Sidewalk: 12,743
  Noise - Vehicle: 11,372

Duplicate unique_key values: 0

Null values per column:
  unique_key: 0
  created_date: 0
  agency_name: 0
  complaint_type: 0
  descriptor: 0
  borough: 0
  incident_zip: 5
  latitude: 779
  longitude: 779


In [16]:
# Export to CSV
output_path = f'{DATA_DIR}/311_complaints.csv'
df_311.to_csv(output_path, index=False)
print(f'Exported {len(df_311):,} rows to {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')

Exported 198,158 rows to ../data/311_complaints.csv
File size: 29659.0 KB


---
## Audit Gate 1 Summary

### Q1 Data (CES)

| Check | Finding | Pass? |
|-------|---------|-------|
| NYC geography identifier | `AREANAME == 'New York City'` — direct entry, NOT the broader MSA | YES |
| Latest available month | **February 2026** (JAN and FEB have data, MAR onward is null) | YES |
| NAICS code format | Data uses BLS industry codes, not NAICS directly. 2-digit NAICS sectors map to BLS industry titles. 18 sectors mapped. | YES |
| NAICS 21+23 combined | Mining (21) and Construction (23) are combined as "Mining, Logging and Construction" — noted as limitation | YES |
| Data is NSA | Confirmed: ces.csv is not seasonally adjusted (per README) | YES |
| COVID baseline for 5yr | Feb 2026 vs Feb 2021 — Feb 2021 is COVID-distorted. Will be noted in analysis. | YES |

### Q2 Data (311 API)

| Check | Finding | Pass? |
|-------|---------|-------|
| Agency names match prompt | Both exact strings found in API: "New York City Police Department" and "Department of Housing Preservation and Development" | YES |
| **HPD has zero matching complaints** | HPD handles housing quality (heat/hot water, plumbing), NOT noise or parking. Zero rows for HPD is CORRECT, not a bug. This is a key finding. | YES |
| **Noise exact type belongs to DEP** | Exact `complaint_type='Noise'` (10,789 records) belongs to Dept of Environmental Protection, NOT NYPD. NYPD handles noise subcategories only. | YES |
| Noise complaint_type values | 9 noise types across all agencies; 6 noise subcategories under NYPD in our filtered data. | YES |
| Noise matching decision | Using `starts_with(complaint_type, 'Noise')` to capture all subcategories. Exact "Noise" is DEP-only and not in our filtered dataset. | YES |
| Data not truncated | 198,158 rows < $limit of 500,000 | YES |
| No duplicates | 0 duplicate unique_key values | YES |
| Date range correct | Min >= 2021-12-15, Max <= 2022-03-15 | YES |
| **All rows are NYPD** | 198,158 rows, 100% NYPD. HPD returns 0. Valid and important finding for Q2b. | YES |

### Open Questions Resolved

| # | Question | Answer |
|---|----------|--------|
| O1 | CES column schema | SERIESCODE, INDUSTRY_TITLE, AREATYPE, AREA, AREANAME, YEAR, JAN-DEC, ANNUAL |
| O2 | NYC area label | `AREANAME == 'New York City'` |
| O3 | Latest available month | **February 2026** |
| O4 | NAICS range codes | Data uses BLS codes; NAICS sectors map to industry titles. No "31-33" format in data. |
| O5 | Noise complaint_type values | 9 values total: Noise (DEP), Noise - Residential/Sidewalk/Vehicle/Commercial/Park/House of Worship (NYPD), Noise - Helicopter (EDC) |
| O6 | Agency name spelling | Exact match confirmed for both agencies |
| O7 | Total 311 rows | **198,158** (all NYPD, 0 HPD) |
| O8 | Missing/suppressed CES data | All 18 mapped industries have valid Feb data for 2021, 2025, 2026 |
